In [ ]:
!pip install segmentation-models-pytorch medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.4 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.nn.functional import silu
from torch.autograd.functional import jacobian
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from collections import Counter
import numpy as np
import os

from medmnist import PneumoniaMNIST

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class RBFMetric(nn.Module):
    def __init__(self,
                 n_clusters = 100,
                 kappa = 1.0,
                 eps=1e-8):
        super().__init__()
        self.n_clusters = n_clusters
        self.kappa = float(kappa)
        self.eps = float(eps)

        self.kmeans = KMeans(n_clusters=self.n_clusters)

        # commented old way:
        #Wdata = (torch.rand(self.n_clusters, 1)**2).to(torch.float16)
        #self.W = nn.Parameter(Wdata, requires_grad=True)

        self.W = nn.Parameter(torch.zeros(self.n_clusters, dtype=torch.float32))

        # set after train_kmeans
        self.cluster_centers = None   # (K, D) torch
        self.lambdas = None           # (K,) torch
        self.cluster_sizes = None
        self.cluster_points_indexes = None

    def _compute_cluster_points_indexes(self):
        #cluster_points_indexes =  [self.kmeans.labels_[self.kmeans.labels_ == k]
        #                                for k in range(self.n_clusters)]
        # return cluster_points_indexes
        labels = self.kmeans.labels_
        return [np.where(labels == k)[0] for k in range(self.n_clusters)]


    def _compute_lambdas(self,data):
        data = torch.tensor(data)
        sum_k = torch.tensor([torch.sum(torch.tensor([torch.linalg.norm(point - self.cluster_centers[k])
                  for point in data[self.cluster_points_indexes[k],:]]))
                     for k in range(self.n_clusters)])
        lambda_k = torch.tensor([torch.pow(self.kappa / self.cluster_sizes[k] * sum_k[k], -2)*0.5
                    for k in range(self.n_clusters)])
        return lambda_k

    def train_kmeans(self, data: np.array):
        print(f"Fitting kmeans")
        self.kmeans.fit(data)
        print(f"Computing auxiliary variables")

        dev = next(self.parameters()).device

        self.cluster_centers = torch.tensor(
            self.kmeans.cluster_centers_ , dtype=torch.float32, requires_grad=False
        )
        self.cluster_sizes = Counter(self.kmeans.labels_)
        self.cluster_points_indexes = self._compute_cluster_points_indexes()

        self.lambdas = self._compute_lambdas(data).to(dtype=torch.float32, device=dev)
        self.cluster_centers = self.cluster_centers.to(device=dev)

        print(f"done")

    def _W(self):
        # softplus -> positive, then normalize to sum to 1
        W_pos = F.softplus(self.W) + self.eps
        W = W_pos / (W_pos.sum() + self.eps)
        return W  # (K,)

    def forward(self, x: torch.Tensor):

        #with torch.no_grad():
        #    self.W.data.clamp(min=0.001)

        #phi_k = torch.stack([torch.exp(-0.5 * self.lambdas[k] * torch.linalg.norm(point - self.cluster_centers[k], dim=1)**2)
        #                        for k in range(self.n_clusters)])

        #metric = torch.mul(self.W, phi_k)

        #metric = torch.sum(metric, dim=0) #(batch_size)

        ##We need this following line to output:
        ##  1 close to data
        ##  0 far from data
        #metric = 1 - torch.abs(1 - metric)

        x = x.to(dtype=torch.float32)

        # dist2: (B, K)
        dist2 = torch.cdist(x, self.cluster_centers, p=2) ** 2

        # phi: (B, K) with broadcasting lambdas (K,)
        phi = torch.exp(-0.5 * dist2 * self.lambdas.unsqueeze(0))

        W = self._W().unsqueeze(0)  # (1, K)

        # convex combination => (B,), bounded in (0,1]
        metric = (phi * W).sum(dim=1)

        return metric

In [ ]:
class Gamma(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()

        self.timestep_embed = nn.Linear(1,2*latent_dim)
        self.mlp = nn.ModuleList([
                                nn.Linear(2*latent_dim, 2*latent_dim),
                                nn.Tanh(),
                                nn.Linear(2*latent_dim, 2*latent_dim),
                                nn.Tanh(),
                                ])
        self.output = nn.Sequential(nn.Linear(2*latent_dim, latent_dim),
                                    nn.Tanh(),
                                    nn.Linear(latent_dim, latent_dim),
                                    )


    def forward(self, x0_latent, x1_latent, timestep):
        t = timestep.unsqueeze(0)
        t = self.timestep_embed(t)
        x = torch.cat((x0_latent, x1_latent), dim=1)
        for _, block in enumerate(self.mlp):
            x = block(x)
        x = x + t
        x = self.output(x)
        return x

In [ ]:
class VectorField(nn.Module):
    def __init__(self, latent_dim=64, hidden_dim = 128, depth=6):
        super().__init__()

        self.timestep_embed = nn.Sequential(nn.Linear(1,hidden_dim),
                                            nn.SiLU(),
                                            nn.Linear(hidden_dim,1))

        self.input_layer = nn.Linear(latent_dim*2 + 1, hidden_dim)

        self.mlp = nn.Sequential()
        for _ in range (depth):
            self.mlp.append(nn.Linear(hidden_dim, hidden_dim),
                            nn.SiLU())

        self.output_layer = nn.Linear(hidden_dim, latent_dim)


    def forward(self, interpolant, timestep):

        batch, _ = x0.size    # TODO fix
        time_embed = self.timestep_embed(timestep).unsqueeze(0).expand(batch, -1)

        interp_time = torch.cat([interpolant, timestep], dim = 1)

        out = self.input_layer(interp_time)
        out = self.mlp(out)
        out = self.output_layer(out)
        out = silu(out)

        return out

---
Load dataset and latents

In [ ]:
img_size=224

data_pneumonia = PneumoniaMNIST(split="train", download=True, size=img_size)
images = torch.tensor(data_pneumonia.imgs, dtype=torch.uint8).to(torch.float16)/255

class CustomImageDataset(Dataset):
    def __init__(self, images):
        #add channel dimension
        self.images = images.unsqueeze(1)
        self.images = self.images.to(device)
        print(f"Dataset shape: {self.images.shape}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx,:,:,:]

images_dataset = CustomImageDataset(images)
batch_size = 256
dataloader = DataLoader(images_dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 214M/214M [00:16<00:00, 12.6MB/s]


Dataset shape: torch.Size([4708, 1, 224, 224])


In [ ]:
######################################################
# Optional: load autoencoder and generate latents
######################################################

#autoencoder = ViTMaskedAutoencoder().cuda()
#autoencoder.load_state_dict(torch.load('autoencoder-mir-epoch30', weights_only=True))
#autoencoder.eval()

# latents = []
# print("------------Generating latents------------")
# for batch in dataloader:
#     latents.append(autoencoder.encode(batch))
# latents = torch.cat(latents).to(dtype=torch.float32)
# latents_numpy = latents.cpu().detach().numpy()
#
# torch.set_default_dtype(torch.float32)
#
# print("---------Latents generated----------------")
# print(f"Generated {latents.shape[0]}latents of dimension {latents.shape[1]}")


##########################
# Load existing Latents
##########################

latents_numpy = np.load("latents_train.npy")          # (N, 256)
labels_numpy  = np.load("labels_train.npy").squeeze() # (N,)

latents = torch.from_numpy(latents_numpy).float().to(device)
labels  = torch.from_numpy(labels_numpy).long().to(device)

# We need a dataset of latents for training the metric
class CustomLatentsDataset(Dataset):
    def __init__(self, latents):
        self.latents = latents

    def __len__(self):
        return len(self.latents)

    def __getitem__(self,idx):
        return self.latents[idx]

---
Train the Metric

In [ ]:
print("--------------Fitting Kmeans---------------")

load_metric = False

# Load all metric related info
if load_metric:
  ckpt = torch.load("metric_full.pt", map_location=device)

  metric = RBFMetric(n_clusters=ckpt["n_clusters"], kappa=ckpt["kappa"]).to(device)
  metric.load_state_dict(ckpt["state_dict"])
  metric.cluster_centers = ckpt["cluster_centers"].to(device=device, dtype=torch.float32)
  metric.lambdas = ckpt["lambdas"].to(device=device, dtype=torch.float32)
  metric.eval()

else:
    metric = RBFMetric().to(device)

    metric.train_kmeans(latents_numpy)

    print("------------ Training metric --------------")

    latents_dataset = CustomLatentsDataset(latents)
    dataloaderLatents = DataLoader(latents_dataset, batch_size=64, shuffle=True)

    optimizer = optim.Adam(metric.parameters())
    n_epochs_metric = 20

    for epoch in range(n_epochs_metric):
        losses = []
        for batch in dataloaderLatents:
            batch = batch.to(device, dtype=torch.float32)
            optimizer.zero_grad()

            m = metric(batch)
            loss = torch.mean((1 - m) ** 2)

            #losses.append(loss.cpu().detach().numpy())   # old way
            losses.append(loss.item())
            loss.backward()
            optimizer.step()

        print(f"Loss at epoch {epoch}: {np.mean(losses)}")
    # torch.save(metric.state_dict(), f"./metric-epoch{epoch}")  # old way of storing

    torch.save({
        "state_dict": metric.state_dict(),          # saves W
        "cluster_centers": metric.cluster_centers.detach().cpu(),
        "lambdas": metric.lambdas.detach().cpu(),
        "n_clusters": metric.n_clusters,
        "kappa": metric.kappa,
    }, "metric_full.pt")


# Metric sanity check

metric.eval()
with torch.no_grad():
    idx = torch.randperm(len(latents))[:512]
    batch = latents[idx]                         # real latents
    score_data = metric(batch).mean().item()

    noise = torch.randn_like(batch)              # random points
    score_noise = metric(noise).mean().item()

print(f"\n[Sanity] metric(data) mean = {score_data:.4f}")  # we want to see sepparation between this two values
print(f"[Sanity] metric(noise) mean = {score_noise:.4f}")


#Up to this point we have the metric, it tells us when a point falls near or
# far of the data distribution

--------------Fitting Kmeans---------------
Fitting kmeans
Computing auxiliary variables
done
------------ Training metric --------------
Loss at epoch 0: 0.8439130710588919
Loss at epoch 1: 0.8393741507787962
Loss at epoch 2: 0.8346354163981773
Loss at epoch 3: 0.8301592066481307
Loss at epoch 4: 0.8256044597239107
Loss at epoch 5: 0.8211963128399205
Loss at epoch 6: 0.8168224265446534
Loss at epoch 7: 0.8125671961823026
Loss at epoch 8: 0.8086057466429633
Loss at epoch 9: 0.8046094324137714
Loss at epoch 10: 0.8005950072327176
Loss at epoch 11: 0.796960104961653
Loss at epoch 12: 0.7932965376892606
Loss at epoch 13: 0.7898116885004817
Loss at epoch 14: 0.7865104498089971
Loss at epoch 15: 0.7832659233260799
Loss at epoch 16: 0.7801374269498361
Loss at epoch 17: 0.7768701596840007
Loss at epoch 18: 0.7739464197609875
Loss at epoch 19: 0.7715380989216469

[Sanity] metric(data) mean = 0.1248
[Sanity] metric(noise) mean = 0.0000


In [ ]:
# Observacio de com ha quedat
metric.eval()
with torch.no_grad():
    idx = torch.randperm(len(latents))[:5000]
    s_data = metric(latents[idx]).detach().cpu()
    s_noise = metric(torch.randn_like(latents[idx])).detach().cpu()

print("data  :", float(s_data.min()), float(s_data.quantile(0.5)), float(s_data.quantile(0.9)), float(s_data.max()))
print("noise :", float(s_noise.min()), float(s_noise.quantile(0.5)), float(s_noise.quantile(0.9)), float(s_noise.max()))

print("lambdas:", float(metric.lambdas.min()), float(metric.lambdas.median()), float(metric.lambdas.max()))

data  : 0.00036290372372604907 0.13409776985645294 0.20431460440158844 0.24574550986289978
noise : 1.339942712341724e-09 6.256329232456892e-09 1.121361048461722e-08 3.115275148957153e-08
lambdas: 0.002514038933441043 0.007826224900782108 36451049472.0


---
Train Gamma

In [ ]:
##########################
# Train Gamma
##########################

#Gamma is the network that will push the interpolants so that they fall
#close to the data distribution, we use the metric to know that

print("-------------Training gamma------------------")

x0_index = (data_pneumonia.labels == 0).squeeze(1)
x1_index = (data_pneumonia.labels == 1).squeeze(1)
latents_x0_dataset = CustomLatentsDataset(latents[x0_index,:])
max_idx = len(latents_x0_dataset)
x1_latents = latents[x1_index,:]
x1_latents = x1_latents[:max_idx,:]
latents_x1_dataset = CustomLatentsDataset(x1_latents)

batch_size=8
x0_dataloader = DataLoader(latents_x0_dataset, batch_size=batch_size, shuffle=True)
x1_dataloader = DataLoader(latents_x1_dataset, batch_size=batch_size, shuffle=True)

timesteps = torch.linspace(0.0, 1.0, len(latents_x1_dataset)).tolist()

#Gamma model
#the gamma model calculates a push towards the good interpolation path
#the good interpolation path is the one where the metric says it is close
#to the distribution given source, target and timestep
_, latent_dim = latents.shape
gamma = Gamma(latent_dim = latent_dim).to(device=device)


#this calculates the derivative of the gamma model,
#it is needed to compute the conditional flow
def func_jacobian(model, x0, x1, timestep_jac):
    def f(timestep_jac_):
        return model(x0, x1, timestep_jac_)
    return jacobian(f, timestep_jac, create_graph=False, vectorize=True)


optimizer = optim.Adam(gamma.parameters())

# TODO load checkpoint of gamma

n_epochs = 20 # TODO update  original 250
losses = []
for epoch in range(n_epochs):
    for i, (x0, x1) in enumerate(zip(x0_dataloader, x1_dataloader)):
        timestep = timesteps[i]

        #first we compute the interpolant at the timestep
        #note that a correction term is added
        gamma_push = gamma(x0,x1,timestep)
        interpolant = timestep*x1 + (1-timestep)*x0 + (timestep*(1-timestep))*gamma_push

        #compute conditional flow
        #this equation is literal what the paper describes
        #you can see how the second and third lines
        #come from the derivative wrt time of the correction term just above
        d_gamma_push = func_jacobian(gamma, x0, x1, timestep)
        conditional_flow = (x1 - x0) / (timesteps[i+1] - timestep) \
                            + (timestep*(1-timestep))* d_gamma_push \
                            + (1 - 2*timestep)*gamma_push

        #measure how deviated from the data is the interpolant
        metric_res = metric.forward(interpolant).unsqueeze(1)

        #compute velocity (is our loss)
        #velocity is the derivative wrt time of the
        velocity = ((conditional_flow**2) * metric_res).sum(dim=-1)
        velocity = velocity.mean()
        #print(f"Velocity: {velocity}")

        optimizer.zero_grad()
        velocity.backward(retain_graph=True)
        torch.nn.utils.clip_grad_norm_(gamma.parameters(), max_norm=1.0)
        optimizer.step()

        velocities.append(velocity.unsqueeze(0))

    loss = torch.cat(velocities)
    losses.append(loss.mean().cpu().detach().numpy())

    print(f"Loss at epoch {epoch}: {losses[-1]}")
    if ((epoch+1)%25 == 0):
        torch.save(gamma.state_dict(), f"./gamma-epoch{epoch:03}")
        torch.save(optimizer.state_dict(), f"./optimizer-gamma-epoch{epoch:03}")

---
Train Vector Field

In [ ]:
# img_size=224
# data_pneumonia = PneumoniaMNIST(split="train", download=True, size=img_size)    # we did it in another cell

healthy_indices = (data_pneumonia.labels == 0).squeeze()
sick_indices = (data_pneumonia.labels == 1).squeeze()
print(data_pneumonia.imgs.shape)
healthy_images = torch.tensor(data_pneumonia.imgs[healthy_indices,:,:], dtype=torch.uint8).float()/255
sick_images = torch.tensor(data_pneumonia.imgs[sick_indices,:,:], dtype=torch.uint8).float()/255
max_idx = len(healthy_images)
sick_images = sick_images[:max_idx,:,:]

# CustomImageDataset implemented in another cell

healthy_images_ds = CustomImageDataset(healthy_images)
sick_images_ds = CustomImageDataset(sick_images)

batch_size=10
healthy_dl = DataLoader(healthy_images_ds, batch_size=batch_size, shuffle=True)
sick_dl = DataLoader(sick_images_ds, batch_size=batch_size, shuffle=True)

In [ ]:
#############
#############
#LOAD YOUR OWN AUTOENCODER
#############
#############

# autoencoder = ViTMaskedAutoencoder().cuda()
# autoencoder.load_state_dict(torch.load('YOUR_PATH', weights_only=True))
# autoencoder.eval()
#
# #Generate healthy latents
# healthy_latents = []
# for batch in healthy_dl:
#     healthy_latents.append(autoencoder.encode(batch))
#
# sick_latents = []
# for batch in sick_dl:
#     sick_latents.append(autoencoder.encode(batch))

#############
#############
#LOAD LATENTS
#############
#############

latents = np.load("latents_train.npy")     # shape: (N, 256)
labels  = np.load("labels_train.npy")      # shape: (N, )

latents = torch.from_numpy(latents).float()
labels  = torch.from_numpy(labels).long()

# Flatten if needed (only if your latents are spatial tensors)
latents = latents.view(latents.shape[0], -1)

# Split healthy vs sick using labels
healthy_latents = latents[labels == 0]
sick_latents    = latents[labels == 1]


#################
#################
# USE GAMMA WITH THE SAME
# LATENT DIMENSIONALITY
# AS YOUR AUTOENCODER
#################
#################

LATENT_DIM = 256

gamma = Gamma(latent_dim = LATENT_DIM).to(device)
gamma.load_state_dict(torch.load("YOUR_PATH"), weights_only=True)
gamma.eval()

vector_field = VectorField(latent_dim = 98)
vector_field.train()

#loss
loss_fn = nn.MSELoss()


#optimizer
optimizer = optim.Adam(vector_field.parameters(), lr=0.001)

#as much timesteps as images
#for bigger dataset more timesteps
timesteps = torch.linspace(0.0, 1.0, len(healthy_images)).tolist()


def func_jacobian(model, x0, x1, timestep_jac):
    def f(timestep_jac_):
        return model(x0, x1, timestep_jac_)
    return jacobian(f, timestep_jac, create_graph=False, vectorize=True)



epochs = 100

for epoch in range(epochs):
    interpolants, conditional_flows = [], []
    for i, (x0, x1) in enumerate(zip(healthy_latents, sick_latents)):

        timestep = timesteps[i]
        gamma_push = Gamma(x0,x1, timestep)

        interpolant = timestep*x1 + (1-timestep)*x0 + (timestep*(1-timestep))*gamma_push
        interpolants.append(interpolant)

        d_gamma_push = func_jacobian(gamma, x0, x1, timestep)

        conditional_flow = (x1 - x0) / (timesteps[i+1] - timestep) \
                                + (timestep*(1-timestep))* d_gamma_push \
                                + (1 - 2*timestep)*gamma_push

        conditional_flows.append(conditional_flow)

    losses = []
    for (interpolant, timestep, conditional_flow) in zip(interpolants, timesteps, conditional_flows):

        vector_field = vector_field(interpolant, timestep)
        loss = loss_fn(vector_field, conditional_flow)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss)

    meanloss = torch.mean(torch.tensor(losses))
    print(f"Loss for epoch {epoch}: {meanloss}")